# 03 — Closed and Discriminative Patterns

This notebook:
1. Loads frequent patterns mined by PrefixSpan.
2. Filters redundant patterns using a closed-pattern criterion.
3. Compares positive and negative support.
4. Produces a compact discriminative pre-sepsis pattern library.

The proposal allows CloSpan or equivalent manual closed-pattern filtering.


In [ ]:
from pathlib import Path
import ast
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PATTERN_DIR = PROJECT_ROOT / "outputs" / "patterns"

positive_df = pd.read_csv(PATTERN_DIR / "frequent_positive_patterns.csv")
negative_df = pd.read_csv(PATTERN_DIR / "frequent_negative_patterns.csv")

positive_df["pattern"] = positive_df["pattern"].apply(ast.literal_eval)
negative_df["pattern"] = negative_df["pattern"].apply(ast.literal_eval)

print("Positive patterns:", len(positive_df))
print("Negative patterns:", len(negative_df))


## Closed-pattern filtering

A pattern is treated as closed within a cohort when no proper superpattern has the same support.

This reduces redundant subpatterns while retaining patterns whose extension changes support.


In [ ]:
def filter_closed_patterns(pattern_df):
    patterns = pattern_df.to_dict("records")
    closed = []

    # Group by support count. A proper superpattern with identical support
    # makes the shorter pattern non-closed.
    for candidate in patterns:
        p = candidate["pattern"]
        support_count = candidate["support_count"]

        is_closed = True

        for other in patterns:
            q = other["pattern"]

            if len(q) <= len(p) or other["support_count"] != support_count:
                continue

            # Ordered subsequence test.
            i = 0
            for item in q:
                if i < len(p) and item == p[i]:
                    i += 1

            if i == len(p):
                is_closed = False
                break

        if is_closed:
            closed.append(candidate)

    return pd.DataFrame(closed)


closed_positive_df = filter_closed_patterns(positive_df)
closed_negative_df = filter_closed_patterns(negative_df)

closed_positive_df.to_csv(PATTERN_DIR / "closed_positive_patterns.csv", index=False)
closed_negative_df.to_csv(PATTERN_DIR / "closed_negative_patterns.csv", index=False)

print("Closed positive patterns:", len(closed_positive_df))
print("Closed negative patterns:", len(closed_negative_df))


## Discriminative pattern selection

Patterns are compared using positive and negative support.

The main principle is:

**high positive support + low negative support = more discriminative pre-sepsis pattern**

The exact threshold is kept configurable.


In [ ]:
NEGATIVE_SUPPORT_FLOOR = 0.20
MIN_POSITIVE_SUPPORT = 0.05

positive_support = dict(
    zip(closed_positive_df["pattern"].astype(str),
        closed_positive_df["support"])
)

negative_support = dict(
    zip(closed_negative_df["pattern"].astype(str),
        closed_negative_df["support"])
)

records = []

for _, row in closed_positive_df.iterrows():
    key = str(row["pattern"])
    pos = float(row["support"])
    neg = float(negative_support.get(key, 0.0))

    if pos >= MIN_POSITIVE_SUPPORT and neg <= NEGATIVE_SUPPORT_FLOOR:
        records.append({
            "pattern": row["pattern"],
            "positive_support": pos,
            "negative_support": neg,
            "support_difference": pos - neg,
            "support_ratio": pos / max(neg, 1e-9),
            "length": row["length"],
        })

discriminative_df = pd.DataFrame(records)

if not discriminative_df.empty:
    discriminative_df = discriminative_df.sort_values(
        ["support_difference", "positive_support"],
        ascending=False
    )

discriminative_df.to_csv(
    PATTERN_DIR / "discriminative_patterns.csv", index=False
)

print("Discriminative patterns:", len(discriminative_df))
display(discriminative_df.head(20))
